# Urgente & Comorbidities EDA — Cluj-Napoca (UAT 54975)

**Goal**: Explore the emergency-visit data (`Urgente_comorbiditati`) filtered
strictly to the municipality of Cluj-Napoca, merge with daily climate indicators,
check UPU vs. urgenta overlap, assess data sufficiency for LSTM modelling, and
produce publication-ready plots.

**Data period**: 2010-2021 (urgenta + nonurgenta), 2014-2021 (UPU).


## 0  Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from constants import (
    URGENTE_DIR, URGENTE_SEP, URGENTE_ENCODING, URGENTE_YEAR_RANGE,
    CLUJ_JUDET_FILTER, CLUJ_LOCALITY_FILTER,
    UPU_COL_REMAP, URGENTE_ONLY_COLS,
    FIRST_PHASE_DIR, SECOND_PHASE_DIR, SECOND_PHASE_FIGURES_DIR,
    UAT_ID_CLUJ_NAPOCA, NUMERIC_COLS,
    FIGURE_WIDTH, FIGURE_HEIGHT, FONT_SIZE, PLT_STYLE,
    WINDOW_SIZES,
)

plt.style.use(PLT_STYLE)
plt.rcParams.update({'font.size': FONT_SIZE})
SECOND_PHASE_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup OK')


## 1  Load all urgente files

Read every `*_urgenta.csv`, `*_nonurgenta.csv`, and `*_UPU.csv` file.
Tag each row with `source_type` and `source_year`.
UPU files have a shifted column layout — we remap them to match the standard
urgenta schema.


In [ ]:
frames = []

# ── UPU normalisation: three schema variants across years ──
# 2014 UPU: 12-col urgenta header, only first 7 fields populated (shifted).
# 2016 UPU (and 2016 urgenta/nonurgenta): 2 junk header lines to skip.
# 2015, 2017-2021 UPU: clean 7-col header (DenJudet, ..., NrFise).
UPU_STANDARD_REMAP = {
    'DenJudet':       'JudetSpital',
    'DenLocalitate':  'LocalitateSpital',
    'DenTipServiciu': 'Criteriu_UPU',
    'DataFisa':       'DataInternare',
    'NrFise':         'NrCazuri',
}

for year in URGENTE_YEAR_RANGE:
    for suffix in ['urgenta', 'nonurgenta', 'UPU']:
        fname = URGENTE_DIR / f'{year}_{suffix}.csv'
        if not fname.exists():
            continue

        # 2016 files have 2 junk lines (``rind`` + dashes) before the real header
        skip = 2 if year == 2016 else 0
        df_tmp = pd.read_csv(fname, sep=URGENTE_SEP, encoding=URGENTE_ENCODING,
                             low_memory=False, skiprows=skip)

        is_upu = (suffix == 'UPU')

        if is_upu:
            if 'DenJudet' in df_tmp.columns:
                # 2015-2021 (except 2014) — proper 7-col UPU header
                df_tmp = df_tmp.rename(columns=UPU_STANDARD_REMAP)
            else:
                # 2014 — 12-col urgenta header with shifted data
                upu_orig = ['JudetSpital', 'LocalitateSpital', 'JudetPacient',
                            'LocalitatePacient', 'Virsta', 'Sex', 'DataInternare']
                upu_real = ['JudetSpital', 'LocalitateSpital', 'Criteriu_UPU',
                            'DataInternare', 'Sex', 'Virsta', 'NrCazuri']
                df_tmp = df_tmp[upu_orig].copy()
                df_tmp.columns = upu_real

            df_tmp['NrCazuri'] = pd.to_numeric(df_tmp['NrCazuri'], errors='coerce').fillna(1).astype(int)
            df_tmp['Virsta']   = pd.to_numeric(df_tmp['Virsta'],   errors='coerce')

        df_tmp['source_type'] = suffix
        df_tmp['source_year'] = year
        frames.append(df_tmp)
        print(f'  {fname.name}: {len(df_tmp):>10,} rows  (UPU={is_upu})')

df_all = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows loaded: {len(df_all):,}')
print(f'Columns: {sorted(df_all.columns.tolist())}')


## 2  Discover & filter Cluj-Napoca municipality

Print all `LocalitateSpital` values under `JudetSpital == 'CLUJ'`, then keep
only the municipality.


In [ ]:
cluj_mask = df_all['JudetSpital'].str.upper() == CLUJ_JUDET_FILTER
print('Localities in Cluj county:')
print(sorted(df_all.loc[cluj_mask, 'LocalitateSpital'].dropna().unique()))


In [ ]:
# Filter strictly to the municipality
muni_mask = (
    (df_all['JudetSpital'] == CLUJ_JUDET_FILTER) &
    (df_all['LocalitateSpital'] == CLUJ_LOCALITY_FILTER)
)
df_cluj = df_all[muni_mask].copy()
print(f'Rows for {CLUJ_LOCALITY_FILTER}: {len(df_cluj):,}')
print(f'Source types: {df_cluj["source_type"].value_counts().to_dict()}')
print(f'Year range : {df_cluj["source_year"].min()} - {df_cluj["source_year"].max()}')


## 3  UPU vs. urgenta overlap check (2014-2021)

For years where both `urgenta` and `UPU` files exist, check whether they share
identical rows (potential double-counting).  We compare on the common columns
`(DataInternare, Sex, Virsta, NrCazuri)`.


In [ ]:
overlap_years = range(2014, 2022)
common_cols = ['DataInternare', 'Sex', 'Virsta', 'NrCazuri']

overlap_report = []
for yr in overlap_years:
    urg = df_cluj[(df_cluj['source_year'] == yr) & (df_cluj['source_type'] == 'urgenta')]
    upu = df_cluj[(df_cluj['source_year'] == yr) & (df_cluj['source_type'] == 'UPU')]
    if upu.empty:
        continue
    # Build hashable tuples for each side (only on common cols)
    urg_set = set(urg[common_cols].dropna().apply(tuple, axis=1))
    upu_set = set(upu[common_cols].dropna().apply(tuple, axis=1))
    overlap = urg_set & upu_set
    overlap_report.append({
        'year': yr,
        'urgenta_rows': len(urg),
        'upu_rows': len(upu),
        'common_tuples': len(overlap),
        'pct_of_upu': round(100 * len(overlap) / max(len(upu_set), 1), 1),
    })

df_overlap = pd.DataFrame(overlap_report)
print(df_overlap.to_string(index=False))
if df_overlap['common_tuples'].sum() == 0:
    print('\n=> No duplicates detected — safe to combine UPU + urgenta.')
else:
    print('\n=> Some overlap detected — review before combining.')


In [ ]:
# Stacked bar: daily totals by source_type per year
df_cluj['DataInternare'] = pd.to_datetime(df_cluj['DataInternare'], errors='coerce')
yearly_src = (
    df_cluj.groupby(['source_year', 'source_type'])['NrCazuri']
    .sum().unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
yearly_src.plot.bar(stacked=True, ax=ax, colormap='Set2')
ax.set_title(f'Annual emergency case totals by source type — {CLUJ_LOCALITY_FILTER}')
ax.set_xlabel('Year')
ax.set_ylabel('Total NrCazuri')
ax.legend(title='source_type')
plt.tight_layout()
fig.savefig(SECOND_PHASE_FIGURES_DIR / '01_annual_cases_by_source.png', dpi=150)
plt.show()


## 4  Aggregate to daily time series

Sum `NrCazuri` per day (across all source types).  Also keep per-source-type
breakdowns and top comorbidity ICD chapter counts.


In [ ]:
df_cluj['date'] = df_cluj['DataInternare'].dt.normalize()

# Total daily cases
daily = (
    df_cluj.groupby('date')['NrCazuri']
    .sum()
    .rename('total_cases')
    .sort_index()
    .to_frame()
)

# Per source_type daily totals (pivot)
daily_src = (
    df_cluj.groupby(['date', 'source_type'])['NrCazuri']
    .sum().unstack(fill_value=0)
)
daily_src.columns = [f'cases_{c}' for c in daily_src.columns]
daily = daily.join(daily_src)

print(f'Daily time series: {len(daily)} days, '
      f'{daily["total_cases"].sum():,.0f} total cases')
print(f'Date range: {daily.index.min().date()} to {daily.index.max().date()}')
print(daily.head())


In [ ]:
# Comorbidity ICD-chapter daily counts (urgenta/nonurgenta only — UPU lacks Comorbiditati)
has_comorb = df_cluj['Comorbiditati'].notna()
print(f'Rows with Comorbiditati field: {has_comorb.sum():,} / {len(df_cluj):,}')

# Extract ICD chapter = first letter of each code
def extract_icd_chapters(comorb_str):
    """Return set of unique ICD chapter letters from a Comorbiditati string."""
    if pd.isna(comorb_str):
        return set()
    chapters = set()
    for part in str(comorb_str).split(','):
        code = part.strip().split('-')[0].strip()
        if code and code[0].isalpha():
            chapters.add(code[0].upper())
    return chapters

# Build per-day ICD chapter counts
icd_records = []
for date_val, grp in df_cluj[has_comorb].groupby('date'):
    chapter_counts = {}
    for _, row in grp.iterrows():
        for ch in extract_icd_chapters(row['Comorbiditati']):
            chapter_counts[ch] = chapter_counts.get(ch, 0) + int(row['NrCazuri'])
    chapter_counts['date'] = date_val
    icd_records.append(chapter_counts)

df_icd = pd.DataFrame(icd_records).set_index('date').fillna(0).astype(int)
df_icd.columns = [f'icd_{c}' for c in df_icd.columns]
daily = daily.join(df_icd)

print(f'ICD chapter columns added: {list(df_icd.columns)}')
print(f'Daily table shape: {daily.shape}')


## 5  Merge with processed climate data

Inner-join the daily emergency counts with `processed_UAT_54975.csv` on `date`.


In [ ]:
climate = pd.read_csv(
    FIRST_PHASE_DIR / f'processed_UAT_{UAT_ID_CLUJ_NAPOCA}.csv',
    parse_dates=['date'],
)
climate = climate.set_index('date')
print(f'Climate rows: {len(climate)}, range: {climate.index.min().date()} – {climate.index.max().date()}')

df_merged = daily.join(climate, how='inner')
print(f'Merged rows (inner): {len(df_merged)}')
print(f'Merged columns: {list(df_merged.columns)}')
df_merged.head()


## 6  EDA plots

### 6a  Daily case-count time series

In [ ]:
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
ax.plot(df_merged.index, df_merged['total_cases'], linewidth=0.5, alpha=0.8)
ax.set_title(f'Daily emergency cases — {CLUJ_LOCALITY_FILTER}')
ax.set_xlabel('Date')
ax.set_ylabel('Total cases')
plt.tight_layout()
fig.savefig(SECOND_PHASE_FIGURES_DIR / '02_daily_cases_timeseries.png', dpi=150)
plt.show()


### 6b  Monthly seasonality

In [ ]:
df_merged['month'] = df_merged.index.month
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
sns.boxplot(data=df_merged, x='month', y='total_cases', ax=ax, palette='coolwarm')
ax.set_title(f'Monthly seasonality of emergency cases — {CLUJ_LOCALITY_FILTER}')
ax.set_xlabel('Month')
ax.set_ylabel('Daily cases')
plt.tight_layout()
fig.savefig(SECOND_PHASE_FIGURES_DIR / '03_monthly_seasonality.png', dpi=150)
plt.show()


### 6c  Yearly totals

In [ ]:
df_merged['year'] = df_merged.index.year
yearly = df_merged.groupby('year')['total_cases'].sum()
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
yearly.plot.bar(ax=ax, color='steelblue')
ax.set_title(f'Annual emergency cases — {CLUJ_LOCALITY_FILTER}')
ax.set_ylabel('Total cases')
plt.tight_layout()
fig.savefig(SECOND_PHASE_FIGURES_DIR / '04_yearly_totals.png', dpi=150)
plt.show()


### 6d  Correlation heatmap (cases vs. climate)

In [ ]:
corr_cols = ['total_cases'] + [c for c in NUMERIC_COLS if c in df_merged.columns]
corr = df_merged[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation: daily emergency cases vs. climate variables')
plt.tight_layout()
fig.savefig(SECOND_PHASE_FIGURES_DIR / '05_correlation_heatmap.png', dpi=150)
plt.show()


### 6e  Scatter plots (cases vs. key climate variables)

In [ ]:
key_vars = ['TX', 'UTCI_MX', 'RH_AVG']
fig, axes = plt.subplots(1, len(key_vars), figsize=(FIGURE_WIDTH, 5))
for ax, var in zip(axes, key_vars):
    if var not in df_merged.columns:
        continue
    ax.scatter(df_merged[var], df_merged['total_cases'], alpha=0.15, s=5)
    ax.set_xlabel(var)
    ax.set_ylabel('Daily cases')
    ax.set_title(f'Cases vs {var}')
plt.suptitle(f'{CLUJ_LOCALITY_FILTER}', fontsize=13)
plt.tight_layout()
fig.savefig(SECOND_PHASE_FIGURES_DIR / '06_scatter_climate.png', dpi=150)
plt.show()


### 6f  Boxplot: heat-stress days vs. normal

In [ ]:
if 'heat_stress_day' in df_merged.columns:
    df_merged['heat_label'] = df_merged['heat_stress_day'].map({True: 'Heat stress', False: 'Normal'})
    fig, ax = plt.subplots(figsize=(8, FIGURE_HEIGHT))
    sns.boxplot(data=df_merged, x='heat_label', y='total_cases', ax=ax, palette='Set1')
    ax.set_title(f'Emergency cases: heat-stress vs. normal days — {CLUJ_LOCALITY_FILTER}')
    ax.set_xlabel('')
    ax.set_ylabel('Daily cases')

    # Print summary stats
    print(df_merged.groupby('heat_label')['total_cases'].describe().round(1))
    plt.tight_layout()
    fig.savefig(SECOND_PHASE_FIGURES_DIR / '07_boxplot_heat_stress.png', dpi=150)
    plt.show()
else:
    print('heat_stress_day column not found in merged data')


### 6g  Top comorbidity ICD chapters

In [ ]:
icd_cols = [c for c in df_merged.columns if c.startswith('icd_')]
if icd_cols:
    icd_totals = df_merged[icd_cols].sum().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
    icd_totals.head(15).plot.bar(ax=ax, color='teal')
    ax.set_title(f'Top ICD chapters in comorbidities — {CLUJ_LOCALITY_FILTER}')
    ax.set_ylabel('Total mentions')
    ax.set_xlabel('ICD chapter code')
    plt.tight_layout()
    fig.savefig(SECOND_PHASE_FIGURES_DIR / '08_top_icd_chapters.png', dpi=150)
    plt.show()
else:
    print('No ICD chapter columns found.')


### 6h  Autocorrelation (ACF / PACF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH, 5))
plot_acf(df_merged['total_cases'].dropna(), lags=60, ax=axes[0], title='ACF — daily cases')
plot_pacf(df_merged['total_cases'].dropna(), lags=60, ax=axes[1], title='PACF — daily cases')
plt.tight_layout()
fig.savefig(SECOND_PHASE_FIGURES_DIR / '09_acf_pacf.png', dpi=150)
plt.show()


## 7  Data sufficiency assessment

Evaluate whether ~12 years of daily data (2010-2021) yields enough training
sequences for each candidate LSTM window size.


In [ ]:
n_days = len(df_merged)
n_train = df_merged[df_merged.index.year <= 2018].shape[0]
n_val   = df_merged[df_merged.index.year == 2019].shape[0]
n_test  = df_merged[df_merged.index.year >= 2020].shape[0]

print(f'Total days: {n_days}')
print(f'Train (<=2018): {n_train}  |  Val (2019): {n_val}  |  Test (>=2020): {n_test}')
print()

MIN_SEQUENCES = 1000
rows = []
for w in WINDOW_SIZES:
    train_seq = max(n_train - w, 0)
    val_seq   = max(n_val - w, 0)
    test_seq  = max(n_test - w, 0)
    ok = 'YES' if train_seq >= MIN_SEQUENCES else 'WARN'
    rows.append({'window': w, 'train_seq': train_seq, 'val_seq': val_seq,
                 'test_seq': test_seq, 'sufficient': ok})

df_suff = pd.DataFrame(rows)
print(df_suff.to_string(index=False))
print()
if (df_suff['sufficient'] == 'WARN').any():
    print('=> Some large windows may be marginal. Consider simpler baselines alongside LSTM.')
else:
    print('=> All window sizes produce enough training sequences.')


## 8  Save merged daily table

In [ ]:
out_path = SECOND_PHASE_DIR / f'urgente_UAT_{UAT_ID_CLUJ_NAPOCA}.csv'
# Drop helper columns before saving
save_cols = [c for c in df_merged.columns if c not in ('month', 'year', 'heat_label')]
df_merged[save_cols].to_csv(out_path)
print(f'Saved {out_path}  ({len(df_merged)} rows, {len(save_cols)} cols)')


## 9  Future Work

1. **Lightweight baseline comparison** — Add an XGBoost or Ridge regression
   baseline as a single reference row in the ablation table (LSTM notebook)
   to anchor the deep-learning results.

2. **COVID-19 sensitivity check** — Flag the COVID lockdown months
   (Mar-May 2020) in all time-series plots. Report test-set metrics both
   with and without those months to quantify pandemic distortion on the
   emergency-visit signal.
